In [2]:
suppressPackageStartupMessages({
    library(jsonlite)
    library(tidyverse)
})

# split each coarse type into chunks 

In [4]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined/', cohort, 'coarsetypes')
cts = c('B', 'Dendritic cell', 'Lining', 'Vascular endothelial', 'Macrophage', 'NK', 'Plasma', 'Sublining', 'T')
params = data.frame(
    ct = cts, 
    cohort = rep(cohort, times = length(cts)), 
    max_mult = rep(1.25, times = length(cts))
    )

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/", 
    paste0("1.split_celltypes_into_chunks_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)

# coarse types to fine types

In [16]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/out_rds', cohort)

params = data.frame()
for (ct in list.dirs(xen_basepath, recursive = FALSE)) {
    chunks = list.files(ct, full.names = TRUE, pattern = "*_untyped.rds")
    lineage_params = data.frame(
        ct = basename(ct), 
        xen_path = chunks, 
        cohort = cohort,
        cca_path = ifelse(basename(ct) %in% c('T', 'Plasma'), # removing B for now as gene expression correlations looked bad
                          paste0('/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/out_ccaweights/', basename(ct), '_ccaweights.RDS'), 
                          ""),
        batch_vars = 'cohort'
        )
    params <- dplyr::bind_rows(params, lineage_params)
}

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/6.fine_types/", 
    paste0("2.celltypes_to_finetypes_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)